# Lab 10: MLP Model for Time-Series Forecasting

**Student Name:** Yousaf Tahir  
**Registration Number:** 22JZELE0479  
**Course:** Machine Learning Lab  
**Supervisor:** Engr. Irshad Ullah  
**University:** UET Nowshera Campus  


## Learning Objectives
- Set the working directory and import required machine learning, TensorFlow, and utility modules.
- Define an MLP neural network architecture for time-series input data.
- Configure optimizer, checkpoints, and training monitor callbacks.
- Load train, validation, and test CSV files for forecasting.
- Train and evaluate the model using MAE, MSE, RMSE, R2, and explained variance score.


## Section 1: Working Directory and Library Imports
This section sets the Lab 10/11 working path and imports Scikit-learn metrics, TensorFlow/Keras layers, callbacks, optimizers, and custom time-series utilities.


In [1]:
import os
os.chdir(r'D:\Machine Learning\ML Lab\Lab 10')

In [2]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
import tensorflow
from tensorflow.keras.layers import Input, Reshape, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
from keras.callbacks import Callback

In [3]:
#lookback = 24
model = None
start_epoch = 0
time_steps=24
num_features=21

In [4]:
def MLP():
    model = Sequential()
    model.add(Flatten(input_shape=(time_steps , num_features)))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1))
    return model

In [5]:
model1 = MLP()
model1.summary()

c:\Users\Yousaf Tahir\anaconda3\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 504)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │        16,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,193 (63.25 KB)

 Trainable params: 16,193 (63.25 KB)

 Non-trainable params: 0 (0.00 B)

## Section 2: Model Parameters and MLP Architecture
The following cells define time steps, number of features, and the MLP model structure used for forecasting.


In [6]:
tensorflow.keras.utils.plot_model(model1 )

You must install pydot (`pip install pydot`) for `plot_model` to work.


In [7]:
checkpoints = r'D:\Machine Learning\ML Lab\Lab 10\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
OUTPUT_PATH = r'D:\Machine Learning\ML Lab\Lab 10'
FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])

In [8]:
os.path.exists(JSON_PATH)

False

In [9]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]

In [10]:
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model =MLP()
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


## Section 3: Checkpoints, Callbacks, and Training Setup
This section configures model checkpoints, training monitor paths, optimizer settings, and compile/training parameters.


In [12]:
import os
path_dataset =r'D:\Machine Learning\ML Lab\Lab 10'
path_tr = os.path.join(path_dataset, 'train.csv')
df_tr = pd.read_csv(path_tr)
train_set = df_tr.iloc[:].values
path_v = os.path.join(path_dataset, 'validation.csv')
df_v = pd.read_csv(path_v)
validation_set = df_v.iloc[:].values 
path_te = os.path.join(path_dataset, 'test.csv')
df_te = pd.read_csv(path_te)
test_set = df_te.iloc[:].values 

path_scaler = os.path.join(path_dataset, 'AEP_scaler.pkl')
scaler = pickle.load(open(path_scaler, 'rb'))

train_set.shape, validation_set.shape, test_set.shape

c:\Users\Yousaf Tahir\anaconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


((860, 21), (90, 21), (30, 21))

In [13]:
start = time.time()
train_X , train_y = univariate_multi_step(train_set, time_steps, target_col=0,target_len=1)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=0,target_len=1)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=0,target_len=1)
print('Time Consumed', time.time()-start, "sec")

Time Consumed 0.0034329891204833984 sec


In [14]:
train_X.shape

(835, 24, 21)

In [15]:
epochs = 60
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                    verbose = verbose)

Epoch 1/60
 1/27 ━━━━━━━━━━━━━━━━━━━━ 17s 660ms/step - loss: 0.3598 - mae: 0.3598 - mape: 137.9386
Epoch 1: val_loss improved from None to 0.06231, saving model to D:\Machine Learning\ML Lab\Lab 10\E1-cp-0001-loss0.06.h5



Epoch 1: finished saving model to D:\Machine Learning\ML Lab\Lab 10\E1-cp-0001-loss0.06.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.1632 - mae: 0.1632 - mape: 80.1163 - val_loss: 0.0623 - val_mae: 0.0623 - val_mape: 18.7063
Epoch 2/60
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0999 - mae: 0.0999 - mape: 32.0019
Epoch 2: val_loss improved from 0.06231 to 0.05801, saving model to D:\Machine Learning\ML Lab\Lab 10\E1-cp-0002-loss0.06.h5



Epoch 2: finished saving model to D:\Machine Learning\ML Lab\Lab 10\E1-cp-0002-loss0.06.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0837 - mae: 0.0837 - mape: 42.5397 - val_loss: 0.0580 - val_mae: 0.0580 - val_mape: 20.2650
Epoch 3/60
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.0734 - mae: 0.0734 - mape: 46.9744
Epoch 3: val_loss did not improve from 0.05801
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0690 - mae: 0.0690 - mape: 39.9181 - val_loss: 0.0834 - val_mae: 0.0834 - val_mape: 27.5543
Epoch 4/60
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.0431 - mae: 0.0431 - mape: 28.7699
Epoch 4: val_loss did not improve from 0.05801
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0690 - mae: 0.0690 - mape: 38.0091 - val_loss: 0.0656 - val_mae: 0.0656 - val_mape: 22.8119
Epoch 5/60
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.1011 - mae: 0.1011 - mape: 254.9201
Epoch 5: val_loss did not improve from 0.05801
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0690 


Epoch 8: finished saving model to D:\Machine Learning\ML Lab\Lab 10\E1-cp-0008-loss0.05.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0563 - mae: 0.0563 - mape: 27.7922 - val_loss: 0.0537 - val_mae: 0.0537 - val_mape: 16.5497
Epoch 9/60
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.0608 - mae: 0.0608 - mape: 21.4274
Epoch 9: val_loss did not improve from 0.05367
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0568 - mae: 0.0568 - mape: 28.6445 - val_loss: 0.0564 - val_mae: 0.0564 - val_mape: 18.4923
Epoch 10/60
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.0354 - mae: 0.0354 - mape: 10.1235
Epoch 10: val_loss did not improve from 0.05367
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0462 - mae: 0.0462 - mape: 22.8103 - val_loss: 0.1120 - val_mae: 0.1120 - val_mape: 38.8564
Epoch 11/60
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.0912 - mae: 0.0912 - mape: 28.7823
Epoch 11: val_loss did not improve from 0.05367
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.056


Epoch 14: finished saving model to D:\Machine Learning\ML Lab\Lab 10\E1-cp-0014-loss0.05.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0540 - mae: 0.0540 - mape: 28.9815 - val_loss: 0.0497 - val_mae: 0.0497 - val_mape: 16.2667
Epoch 15/60
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.0402 - mae: 0.0402 - mape: 12.0981
Epoch 15: val_loss did not improve from 0.04973
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0405 - mae: 0.0405 - mape: 20.0557 - val_loss: 0.0563 - val_mae: 0.0563 - val_mape: 17.5303
Epoch 16/60
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 0.0647 - mae: 0.0647 - mape: 30.3298
Epoch 16: val_loss did not improve from 0.04973
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0602 - mae: 0.0602 - mape: 30.1953 - val_loss: 0.0533 - val_mae: 0.0533 - val_mape: 17.1092
Epoch 17/60
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.0371 - mae: 0.0371 - mape: 26.5946
Epoch 17: val_loss improved from 0.04973 to 0.04048, saving model to D:\Machine Learning\ML Lab\L


Epoch 17: finished saving model to D:\Machine Learning\ML Lab\Lab 10\E1-cp-0017-loss0.04.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0498 - mae: 0.0498 - mape: 24.6434 - val_loss: 0.0405 - val_mae: 0.0405 - val_mape: 13.0794
Epoch 18/60
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.0334 - mae: 0.0334 - mape: 10.9010
Epoch 18: val_loss did not improve from 0.04048
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0374 - mae: 0.0374 - mape: 18.3587 - val_loss: 0.0524 - val_mae: 0.0524 - val_mape: 18.2301
Epoch 19/60
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.0326 - mae: 0.0326 - mape: 10.6840
Epoch 19: val_loss improved from 0.04048 to 0.03854, saving model to D:\Machine Learning\ML Lab\Lab 10\E1-cp-0019-loss0.04.h5



Epoch 19: finished saving model to D:\Machine Learning\ML Lab\Lab 10\E1-cp-0019-loss0.04.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0385 - mae: 0.0385 - mape: 19.0874 - val_loss: 0.0385 - val_mae: 0.0385 - val_mape: 12.4646
Epoch 20/60
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.0309 - mae: 0.0309 - mape: 13.2601
Epoch 20: val_loss did not improve from 0.03854
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0335 - mae: 0.0335 - mape: 19.5699 - val_loss: 0.0392 - val_mae: 0.0392 - val_mape: 11.5752
Epoch 21/60
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.0350 - mae: 0.0350 - mape: 13.1060
Epoch 21: val_loss did not improve from 0.03854
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0431 - mae: 0.0431 - mape: 21.0205 - val_loss: 0.0459 - val_mae: 0.0459 - val_mape: 16.3140
Epoch 22/60
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.0419 - mae: 0.0419 - mape: 14.4775
Epoch 22: val_loss did not improve from 0.03854
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.


Epoch 45: finished saving model to D:\Machine Learning\ML Lab\Lab 10\E1-cp-0045-loss0.03.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0349 - mae: 0.0349 - mape: 16.2079 - val_loss: 0.0339 - val_mae: 0.0339 - val_mape: 10.0437
Epoch 46/60
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0329 - mae: 0.0329 - mape: 12.7789
Epoch 46: val_loss did not improve from 0.03387
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0373 - mae: 0.0373 - mape: 19.8820 - val_loss: 0.0612 - val_mae: 0.0612 - val_mape: 20.1775
Epoch 47/60
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.0386 - mae: 0.0386 - mape: 12.8680
Epoch 47: val_loss did not improve from 0.03387
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0342 - mae: 0.0342 - mape: 20.5002 - val_loss: 0.0471 - val_mae: 0.0471 - val_mape: 12.6986
Epoch 48/60
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.0306 - mae: 0.0306 - mape: 16.8225
Epoch 48: val_loss did not improve from 0.03387
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.

In [16]:
model = load_model(r'D:\Machine Learning\ML Lab\Lab 10\E1-cp-0045-loss0.03.h5', compile=False)

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
Mean Absolute Error (MAE): 740.81
Median Absolute Error (MedAE): 795.21
Mean Squared Error (MSE): 730659.8
Root Mean Squared Error (RMSE): 854.79
Mean Absolute Percentage Error (MAPE): 4.74 %
Median Absolute Percentage Error (MDAPE): 5.06 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


## Section 4: Dataset Loading and Forecast Evaluation
The remaining cells load CSV datasets, prepare forecasting windows, run prediction, and evaluate model performance using regression metrics.


# Fine Tuning

In [23]:
checkpoints = r'D:\Machine Learning\ML Lab\Lab 10\\E2-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
model=r'D:\Machine Learning\ML Lab\Lab 10\E1-cp-0045-loss0.03.h5\E1-cp-0055-loss0.03.h5'
start_epoch= 56

In [24]:
# construct the callback to save only the best model to disk
EpochCheckpoint1 = ModelCheckpoint(
    checkpoints,
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

TrainingMonitor1 = TrainingMonitor(
    FIG_PATH,
    jsonPath=JSON_PATH,
    startAt=start_epoch
)

callbacks = [EpochCheckpoint1, TrainingMonitor1]

model_path = r'D:\Machine Learning\ML Lab\Lab 10\E1-cp-0045-loss0.03.h5'
# model_path = None   # use this if you want to train from scratch

if model_path is None:
    print("[INFO] compiling model...")
    model = PC.build(time_steps=24, num_features=21, reg=0.0005)

    opt = Adam(learning_rate=1e-3)
    model.compile(
        loss="mae",
        optimizer=opt,
        metrics=["mae", "mape"]
    )

else:
    print("[INFO] loading {}...".format(model_path))
    model = load_model(model_path, compile=False)

    opt = Adam(learning_rate=1e-4)
    model.compile(
        loss="mae",
        optimizer=opt,
        metrics=["mae", "mape"]
    )

    print("[INFO] learning rate: {}".format(K.get_value(model.optimizer.learning_rate)))

[INFO] loading D:\Machine Learning\ML Lab\Lab 10\E1-cp-0045-loss0.03.h5...
[INFO] learning rate: 9.999999747378752e-05


In [25]:
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                        verbose = verbose)

Epoch 1/10


 1/27 ━━━━━━━━━━━━━━━━━━━━ 13s 526ms/step - loss: 0.0350 - mae: 0.0350 - mape: 32.9010
Epoch 1: val_loss improved from None to 0.03790, saving model to D:\Machine Learning\ML Lab\Lab 10\\E2-cp-0001-loss0.04.h5



Epoch 1: finished saving model to D:\Machine Learning\ML Lab\Lab 10\\E2-cp-0001-loss0.04.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0224 - mae: 0.0224 - mape: 10.5835 - val_loss: 0.0379 - val_mae: 0.0379 - val_mape: 10.1626
Epoch 2/10
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0231 - mae: 0.0231 - mape: 6.2908
Epoch 2: val_loss did not improve from 0.03790
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0208 - mae: 0.0208 - mape: 8.9070 - val_loss: 0.0446 - val_mae: 0.0446 - val_mape: 13.0990
Epoch 3/10
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0196 - mae: 0.0196 - mape: 5.8012
Epoch 3: val_loss did not improve from 0.03790
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0197 - mae: 0.0197 - mape: 8.6214 - val_loss: 0.0423 - val_mae: 0.0423 - val_mape: 12.0735
Epoch 4/10
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0172 - mae: 0.0172 - mape: 7.0372
Epoch 4: val_loss improved from 0.03790 to 0.03779, saving model to D:\Machine Learning\ML Lab\Lab 10\\E2-


Epoch 4: finished saving model to D:\Machine Learning\ML Lab\Lab 10\\E2-cp-0004-loss0.04.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0199 - mae: 0.0199 - mape: 8.8785 - val_loss: 0.0378 - val_mae: 0.0378 - val_mape: 10.7654
Epoch 5/10
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.0182 - mae: 0.0182 - mape: 5.7300
Epoch 5: val_loss did not improve from 0.03779
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0199 - mae: 0.0199 - mape: 8.4516 - val_loss: 0.0392 - val_mae: 0.0392 - val_mape: 11.3105
Epoch 6/10
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.0152 - mae: 0.0152 - mape: 6.3117
Epoch 6: val_loss did not improve from 0.03779
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0193 - mae: 0.0193 - mape: 8.0188 - val_loss: 0.0428 - val_mae: 0.0428 - val_mape: 12.8140
Epoch 7/10
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 0.0205 - mae: 0.0205 - mape: 8.6106
Epoch 7: val_loss did not improve from 0.03779
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0187 - mae: 

In [26]:
model = load_model(r'D:\Machine Learning\ML Lab\Lab 10\\E2-cp-0004-loss0.04.h5', compile=False)

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
Mean Absolute Error (MAE): 688.51
Median Absolute Error (MedAE): 535.13
Mean Squared Error (MSE): 674808.56
Root Mean Squared Error (RMSE): 821.47
Mean Absolute Percentage Error (MAPE): 4.42 %
Median Absolute Percentage Error (MDAPE): 3.4 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


## Final Conclusion
In this lab, an MLP neural network was implemented for time-series forecasting. The notebook demonstrates model creation, callback configuration, training workflow, and regression-based evaluation.